# 04 - Treinamento do Modelo
## XGBoost Regressor com Walk-Forward Validation
Treinamos um modelo para prever o retorno futuro de 21 dias de cada ação.
A avaliação é feita pela qualidade do ranking: as ações que o modelo
coloca no topo realmente performam melhor que as do fundo?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("../data/processed/features_with_regime.parquet")
print(f"Shape: {df.shape}")
print(f"Período: {df.index.get_level_values('date').min()} até {df.index.get_level_values('date').max()}")

### 4.1 Definição das features e target
Separamos as colunas que o modelo vai usar como entrada (features)
e a coluna que ele vai tentar prever (target).
Excluímos colunas que causariam data leakage (informação do futuro)
e colunas descritivas.

In [ ]:
exclude_cols = [
    "open", "high", "low", "close", "adj_close", "volume",
    "target_21d",
    "dollar_volume", "obv",
    "macd", "macd_signal",
    "market_regime_name"
]

feature_cols = [c for c in df.columns if c not in exclude_cols]
target_col = "target_21d"

print(f"Features ({len(feature_cols)}):")
print(feature_cols)
print(f"\nTarget: {target_col}")

### 4.2 Instalação do XGBoost
Importamos o XGBoost. Se não estiver instalado, a célula abaixo instala.

In [ ]:
try:
    import xgboost as xgb
    print(f"XGBoost versão: {xgb.__version__}")
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "xgboost"])
    import xgboost as xgb
    print(f"XGBoost instalado. Versão: {xgb.__version__}")

### 4.3 Walk-Forward Validation
Simulamos o uso real do modelo: a cada mês, treinamos com todos os dados
disponíveis até aquele momento e prevemos o mês seguinte.
Usamos no mínimo 2 anos de dados para o primeiro treino.

Janela expansiva: o conjunto de treino cresce a cada iteração,
incorporando mais dados históricos.

In [ ]:
dates = df.index.get_level_values("date")
unique_dates = sorted(dates.unique())

monthly_ends = pd.Series(unique_dates).groupby(
    pd.Series(unique_dates).dt.to_period("M")
).last().values

min_train_days = 504  # ~2 anos de pregão
train_start_idx = min_train_days

monthly_ends_filtered = [d for d in monthly_ends if d >= unique_dates[train_start_idx]]

print(f"Total de datas únicas: {len(unique_dates)}")
print(f"Rebalanceamentos mensais: {len(monthly_ends_filtered)}")
print(f"Primeiro rebalanceamento: {monthly_ends_filtered[0]}")
print(f"Último rebalanceamento: {monthly_ends_filtered[-1]}")

### 4.4 Treinamento walk-forward
Para cada mês:
1. Treina o XGBoost com todos os dados até o final do mês anterior
2. Prevê os retornos de todas as ações no mês seguinte
3. Armazena as previsões para avaliação posterior

In [ ]:
xgb_params = {
    "n_estimators": 200,
    "max_depth": 5,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "random_state": 42,
    "n_jobs": -1
}

all_predictions = []

for i in range(len(monthly_ends_filtered) - 1):
    train_end = monthly_ends_filtered[i]
    test_start = monthly_ends_filtered[i]
    test_end = monthly_ends_filtered[i + 1]

    train_mask = dates <= train_end
    test_mask = (dates > test_start) & (dates <= test_end)

    train_data = df.loc[train_mask].dropna(subset=[target_col])
    test_data = df.loc[test_mask].dropna(subset=[target_col])

    if len(train_data) < 100 or len(test_data) < 10:
        continue

    X_train = train_data[feature_cols]
    y_train = train_data[target_col]
    X_test = test_data[feature_cols]
    y_test = test_data[target_col]

    model = xgb.XGBRegressor(**xgb_params)
    model.fit(X_train, y_train, verbose=False)

    preds = model.predict(X_test)

    pred_df = pd.DataFrame({
        "date": test_data.index.get_level_values("date"),
        "ticker": test_data.index.get_level_values("ticker"),
        "predicted_return": preds,
        "actual_return": y_test.values
    })
    all_predictions.append(pred_df)

    if (i + 1) % 12 == 0:
        elapsed = i + 1
        total = len(monthly_ends_filtered) - 1
        print(f"Progresso: {elapsed}/{total} meses processados")

predictions = pd.concat(all_predictions, ignore_index=True)
print(f"\nConcluído. Total de previsões: {len(predictions):,}")
print(f"Período: {predictions['date'].min()} até {predictions['date'].max()}")

### 4.5 Avaliação: correlação entre previsão e realidade
A correlação de Spearman mede se a ordem (ranking) das previsões
corresponde à ordem dos retornos reais. Não importa se o modelo
erra o valor exato, importa se ele acerta quem vai performar melhor.

In [ ]:
from scipy.stats import spearmanr

monthly_corrs = []

for date, group in predictions.groupby("date"):
    if len(group) < 10:
        continue
    corr, _ = spearmanr(group["predicted_return"], group["actual_return"])
    monthly_corrs.append({"date": date, "spearman_corr": corr})

corr_df = pd.DataFrame(monthly_corrs)

print(f"Correlação de Spearman média: {corr_df['spearman_corr'].mean():.4f}")
print(f"Correlação de Spearman mediana: {corr_df['spearman_corr'].median():.4f}")
print(f"% de meses com correlação positiva: {(corr_df['spearman_corr'] > 0).mean():.1%}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(corr_df["date"], corr_df["spearman_corr"],
       color=corr_df["spearman_corr"].apply(lambda x: "green" if x > 0 else "red"),
       width=20, alpha=0.7)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.axhline(y=corr_df["spearman_corr"].mean(), color="blue", linewidth=1, linestyle="--",
           label=f"Média: {corr_df['spearman_corr'].mean():.4f}")
ax.set_ylabel("Correlação de Spearman")
ax.set_title("Correlação entre ranking previsto e ranking real (por dia)")
ax.legend()
plt.tight_layout()
plt.savefig("../results/spearman_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.6 Avaliação: retorno dos portfólios por quintil
Dividimos as ações em 5 grupos (quintis) pelo retorno previsto.
Se o modelo funciona, o quintil superior (Q5) deve ter retorno real
maior que o quintil inferior (Q1).

In [ ]:
predictions["quintile"] = predictions.groupby("date")["predicted_return"].transform(
    lambda x: pd.qcut(x, 5, labels=[1, 2, 3, 4, 5], duplicates="drop")
)
predictions["quintile"] = predictions["quintile"].astype(int)

quintile_returns = predictions.groupby("quintile")["actual_return"].mean()

print("Retorno médio real por quintil de previsão:")
print(quintile_returns.round(6))
print(f"\nSpread Q5 - Q1: {quintile_returns[5] - quintile_returns[1]:.6f}")

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#d32f2f", "#ff9800", "#9e9e9e", "#4caf50", "#1b5e20"]
quintile_returns.plot(kind="bar", ax=ax, color=colors)
ax.set_xlabel("Quintil (1=pior previsão, 5=melhor previsão)")
ax.set_ylabel("Retorno médio real (21 dias)")
ax.set_title("Retorno real por quintil de previsão do modelo")
ax.axhline(y=0, color="black", linewidth=0.5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../results/quintile_returns.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.7 Feature importance
Quais features o modelo considerou mais importantes para fazer as previsões.

In [ ]:
importance = model.feature_importances_
feat_importance = pd.Series(importance, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 10))
feat_importance.tail(20).plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 20 features mais importantes (último modelo treinado)")
ax.set_xlabel("Importância")
plt.tight_layout()
plt.savefig("../results/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

### 4.8 Salvar previsões

In [ ]:
predictions.to_parquet("../data/processed/predictions.parquet", index=False)
print("Salvo em data/processed/predictions.parquet")
print(f"Shape: {predictions.shape}")